# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides you through loading, overviewing, and analyzing the FAIR^2 dataset using the `mlcroissant` library. All references to fields, record sets, and columns use their `@id`s as defined in the Croissant schema.

### Dataset Source
Dataset Croissant schema URL:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and prepare for exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Review the available record sets and their fields using their `@id`s.

In [ ]:
# List available RecordSets and their @id
record_sets_info = []
for rs in metadata.recordSet:
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', None)
    fields = getattr(rs, 'field', [])
    field_ids = [getattr(f, '@id', None) for f in fields]
    record_sets_info.append({'id': rs_id, 'name': rs_name, 'fields': field_ids})
    print(f"RecordSet @id: {rs_id} | name: {rs_name}")
    print(f"  Fields @id: {field_ids}\n")

# For demonstration, list the first 5 records from the first RecordSet
if record_sets_info:
    rs0_id = record_sets_info[0]['id']
    for i, record in enumerate(dataset.records(record_set=rs0_id)):
        print(f"Record {i+1}", record)
        if i >= 4:
            break
else:
    print("No record sets defined in the dataset.")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames using their `@id`s.

In [ ]:
# Extract data from each record set as DataFrames, referencing all IDs by their @id
dataframes = {}

# Gather record set @ids
record_sets_ids = [info['id'] for info in record_sets_info]

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for RecordSet '{record_set_id}' with columns:")
    print(dataframes[record_set_id].columns.tolist())
    print(dataframes[record_set_id].head(), '\n')

# Choose a main record set for demonstration:
# If multiples, pick the first one by default
main_recordset_id = record_sets_ids[0] if record_sets_ids else None

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group records by fields using their `@id`s.

In [ ]:
# Find a numeric field in the main record set to demonstrate filtering and normalization
df = dataframes[main_recordset_id] if main_recordset_id else None

# Attempt to identify a numeric field automatically
numeric_field_id = None
if df is not None:
    for col in df.columns:
        # Try numeric conversion for sample values
        sample = df[col].dropna().head(10)
        try:
            vals = pd.to_numeric(sample)
            if vals.notna().sum() > 0:
                numeric_field_id = col
                break
        except Exception:
            continue
else:
    print("No data loaded.")

if df is not None and numeric_field_id:
    # Threshold for demonstration
    threshold = df[numeric_field_id].astype(float).mean() if not pd.isnull(df[numeric_field_id].astype(float).mean()) else 10
    filtered_df = df[df[numeric_field_id].astype(float) > threshold]
    print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
    print(filtered_df.head())

    # Normalization
    norm_col_id = f"{numeric_field_id}_normalized"
    filtered_df[norm_col_id] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col_id]].head())

    # Try grouping by a categorical field
    # Prefer the first non-numeric column as grouping field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id:
            vals = df[col].dropna().head(10)
            types = set(type(val).__name__ for val in vals)
            if 'str' in types or len(vals.unique()) < 20:
                group_field_id = col
                break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped '{numeric_field_id}' by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("Could not identify a numeric field for filtering and normalization.")

## 5. Visualization
Visualize numeric distributions and relationships between fields.

In [ ]:
if df is not None and numeric_field_id:
    plt.figure(figsize=(8,4))
    plt.hist(df[numeric_field_id].astype(float).dropna(), bins=20, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field available, visualize mean by group
    if group_field_id:
        means = df.groupby(group_field_id)[numeric_field_id].mean().dropna()
        means.plot(kind='bar', color='lightcoral')
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, you explored the Ordered Logistic Regression dataset by loading its Croissant metadata, examining available record sets and fields using `@id`s, extracting data for analysis, and visualizing key variable distributions and relationships. All processing and references used the unique entity identifiers as required for FAIR data practices. This systematic workflow enables scalable and reproducible exploration for scientific or policy insight.